In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas
from musicdb import PanDBIO, PanDBMetrics
from gate import IOStore
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
hio    = HTMLIO()
io     = FileIO()
mp     = MasterParams(verbose=True)
dbs    = mp.getDBs()
pdbio  = PanDBIO()
ios    = IOStore()
mdbios = ios.get()

# Better Counts

In [ ]:
pdbm = PanDBMetrics()
pdbm.addMetrics()

In [ ]:
pdbio = PanDBIO()
pdbio.setIndex()

In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()
mmeDF

# Born/Formed/Country

In [ ]:
summaryType = "Bio"
bioData  = {}
ios = IOStore()
mdbios = ios.get()
for db,mdbio in mdbios.items():
    print(f"{db: <30}".format(db),end="")
    df = eval("mdbio.data.getSummary{0}Data()".format(summaryType))
    if isinstance(df,DataFrame):
        bioData[db] = df
    else:
        print("")
        continue
    print(bioData[db].shape)

##########################################################################################################################################################
##  Notes:
##########################################################################################################################################################
# Discogs: RealName, but not very useful
# RateYourMusic: ['Born', 'Formed', 'Currently', 'Disbanded', 'Notes']
# MyMusicTapez: Description (descriptive, but not very useful)
# Deezer: Image
# MetalArchives: ['Formed', 'Location', 'Country', 'Themes']
# MusicBrainz: ['Country', 'ActiveDates']

## Country

In [ ]:
from collections import Counter
def flattenCountryList(row):
    retval = Counter()
    for rowElem in row:
        if isinstance(rowElem,list):
            for item in rowElem:
                if isinstance(item,str):
                    retval[item] += 1
        elif isinstance(rowElem,str):
            retval[rowElem] += 1
            
    retval = retval.most_common()[0][0] if len(retval) > 0 else None
    return retval

def getRYMCountry(row):
    retval = [country for country in row.unique() if isinstance(country,str)]
    retval = retval if len(retval) > 0 else None
    return retval

ts = Timestat("Finding Pandb Country")
countryData = {}
bioData = ios.get("RateYourMusic").data.getSummaryBioData()
countryData["RateYourMusic"] = bioData.applymap(lambda item: item.get("Country") if isinstance(item,dict) else None).apply(getRYMCountry, axis=1)
ts.update(cmt="RateYourMusic")

bioData = ios.get("MusicBrainz").data.getSummaryBioData()
countryData["MusicBrainz"]   = bioData["Country"].apply(lambda item: [item] if isinstance(item,str) else None)
ts.update(cmt="MusicBrainz")

bioData = ios.get("MetalArchives").data.getSummaryBioData()
countryData["MetalArchives"] = bioData["Country"].apply(lambda item: [item] if isinstance(item,str) else None)
ts.update(cmt="MetalArchives")

pdbCountryData = concat({db: mmeDF[db].map(countryData[db]) for db,dbCountryData in countryData.items()}, axis=1)
pdbCountry     = pdbCountryData.apply(flattenCountryList, axis=1)
ts.stop()

## Dates

In [ ]:
summaryType = "Dates"
datesData  = {}
ios = IOStore()
mdbios = ios.get()
for db,mdbio in mdbios.items():
    print(f"{db: <30}".format(db),end="")
    df = eval("mdbio.data.getSummary{0}Data()".format(summaryType))
    if isinstance(df,DataFrame):
        datesData[db] = df
    else:
        print("")
        continue
    print(datesData[db].shape)

In [ ]:
# Spotify, Beatport, Traxsource all have the media date stored in 'aformat'
# MusicBrainz seems to be missing that data for now

In [ ]:

ts = Timestat("Finding Pandb Year")
yearData = {}

yearData["Discogs"] = ios.get("Discogs").data.getSummaryDatesData()
ts.update(cmt="Discogs")

yearData["RateYourMusic"] = ios.get("RateYourMusic").data.getSummaryDatesData()
ts.update(cmt="RateYourMusic")

yearData["MetalArchives"] = ios.get("MetalArchives").data.getSummaryDatesData()
ts.update(cmt="MetalArchives")

yearData["AlbumOfTheYear"] = ios.get("AlbumOfTheYear").data.getSummaryDatesData()
ts.update(cmt="AlbumOfTheYear")

yearData["JioSaavn"] = ios.get("JioSaavn").data.getSummaryDatesData()
ts.update(cmt="JioSaavn")

pdbio = PanDBIO()
mmeDF = pdbio.getData()
pdbYear = {}
for yearType in ["MinDate", "MaxYear", "MedianYear"]:
    pdbYearData = concat({db: mmeDF[db].map(yearData[db][yearType]) for db,dbYearData in yearData.items()}, axis=1)
    pdbYear[yearType] = pdbYearData.mean(axis=1).round(0)
pdbYear = concat(pdbYear, axis=1).rename(columns={"MinDate": "MinYear"})
ts.stop()

In [ ]:
pdbYear

In [ ]:
for db,dd in datesData.items():
    counts = dd['MedianYear'].count()
    print(f"{db: <20}{counts}")

In [ ]:
modValData = mdbios["MusicBrainz"].data.getModValData("33")

In [ ]:
modValData['196443147811866574813694224356456169333'].media.media["Album + Compilation"][0].get()

In [ ]:
from master import MasterMetas
mm = MasterMetas()
mm.getMetas()
from lib import rateyourmusic
mdbio = rateyourmusic.MusicDBIO(verbose=True)

In [ ]:
mdbios["RateYourMusic"].meta.make(modVal=0, metatype="Dates", verbose=True)

In [ ]:
mediaDates = mdbios["RateYourMusic"].data.getMetaDatesData(0)

In [ ]:
from statistics import median
from listUtils import getFlatList

def getMediaDateStats(mediaDates):
    mediaTypeDates  = {mediaType: [int(year) for year in mediaTypeYears] for mediaType,mediaTypeYears in mediaDates.items() if len(mediaTypeYears) > 0}
    mediaTypeDates  = getFlatList(mediaTypeDates.values())
    mediaDatesStats = (min(mediaTypeDates), max(mediaTypeDates), int(median(mediaTypeDates))) if len(mediaTypeDates) > 0 else (None,None,None)
    return mediaDatesStats

tmp = mediaDates["Dates"].head(10).apply(getMediaDateStats)

In [ ]:
mediaDates

In [ ]:
bioData['RateYourMusic']

In [ ]:
rymbioData = {}
for col in ['Born', 'Formed', 'Currently', 'Disbanded']: #, 'Notes']
    for key in ['Date', 'City', 'State', 'Country']:
        rymbioData[(col,key)] = bioData['RateYourMusic'][col].apply(lambda x: x.get(key) if isinstance(x,dict) else None)
rymbioData = DataFrame(rymbioData)

In [ ]:
def getSingleDate(row):
    dateValues = []
    for dateType in ['Disbanded', 'Formed', 'Born', 'Currently']:
        dateValue = row[(dateType,'Date')]
        if isinstance(dateValue,(int,str)):
            dateValues.append(int(dateValue))
    retval = [min(dateValues),max(dateValues)] if len(dateValues) > 0 else [None,None]
    return retval
    
singleDate = rymbioData.apply(getSingleDate, axis=1)

In [ ]:
modValData = mdbios["RateYourMusic"].data.getModValData(0)

In [ ]:
{mediaType: [mediaData.year for mediaData in mediaTypeData if isinstance(mediaData.year,str)] for mediaType,mediaTypeData in modValData['1600'].media.media.items()}

In [ ]:
notes = bioData['RateYourMusic']['Notes']
notes[notes.notna()].apply(lambda x: x[0].get())

In [ ]:
from pandas import Series, isna
def getMapValue(dbIDKey, lookup):
    if isinstance(lookup,Series):
        if isinstance(dbIDKey,str):
            return lookup.get(dbIDKey)
        elif isinstance(dbIDKey,list):
            return [lookup.get(key) for key in dbIDKey]
        elif isna(dbIDKey):
            return None
        else:
            print(dbIDKey)
            raise ValueError("Not sure how to parse {0}".format(type(dbIDKey)))
    else:
        raise ValueError("Not sure how to use lookup {0}".format(type(lookup)))

In [ ]:
summaryData = {}
for sumType in sumTypes:
    ts = Timestat("Getting {0} Data".format(sumType))
    sumTypeData = {db: eval("mdbio.data.getSummary{0}Data".format(sumType))() for db,mdbio in mdbios.items()}
    ts.update()
    sumTypeData = {db: dbSumTypeData for db,dbSumTypeData in sumTypeData.items() if isinstance(dbSumTypeData,DataFrame)}
    ts.update()

    sumTypeDataDF = {}
    for db,dbID in mmeDF.items():
        ts.update()
        dbSumTypeData = sumTypeData.get(db)
        if isinstance(dbSumTypeData,DataFrame):
            for key,value in dbSumTypeData.items():
                gkey = "_".join([db,key])
                sumTypeDataDF[gkey] = dbID.apply(lambda dbIDKey: getMapValue(dbIDKey, value))
    sumTypeDataDF = DataFrame(sumTypeDataDF)
    ts.stop()
    summaryData[sumType] = sumTypeDataDF

In [ ]:
from ioutils import FileIO
from utils import MusicDBPermDir
mdbpd = MusicDBPermDir()
io = FileIO()
for sumType,sumTypeDataDF in summaryData.items():
    ifile = mdbpd.getMatchPermPath().join("summary{0}.p".format(sumType))
    io.save(idata=sumTypeDataDF, ifile=ifile.str)

In [ ]:
from utils import MusicDBPermDir
mdbpd = MusicDBPermDir()
sumType = "Counts"
ifile = mdbpd.getMatchPermPath().join("summary{0}.p".format(sumType))
countsData = io.get(ifile)